# 05 답변 평가와 평가자 검증

목표: 근거 기반 판정과 평가자 오류 분석. judge 결과는 교육용 작성 fixture이며 실제 모델 호출이나 인간 주석가 연구가 아닙니다.

**준비:** 저장소 터미널에서 python -m pip install -r requirements/week11-offline.txt 실행.
VS Code 커널에서 저장소 .venv의 **Python 3.11** 선택 후 위에서 아래로 실행합니다.
각 파일은 다른 Notebook의 변수를 요구하지 않습니다. GPU·Ollama·API 키는 필요 없습니다.

**데이터:** 교수 작성 가상 쇼핑 정책입니다. 실제 기업 규정이나 고객 기록이 아닙니다.
핵심 단계는 수업에서 함께 실행하고 확장 실험은 복습용입니다. [실습 안내](../README.md).


In [ ]:
from pathlib import Path
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week11/lab/evaluation.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("저장소 안에서 Notebook을 실행하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
import pandas as pd
from IPython.display import display
from week11.lab.evaluation import load_data

docs, questions = load_data()
dev = [q for q in questions if q["split"] == "dev"]
test = [q for q in questions if q["split"] == "test"]
print("환경 준비 완료", sys.version.split()[0], "문서", len(docs), "질문", len(questions))

## 1. 판정 기준

LLM-as-a-Judge는 모델이 답변을 평가하는 방식입니다. 유창성이 사실성을 대신하지 못합니다.

In [ ]:
from week11.lab.judging import (
    RUBRIC,
    FIXTURES,
    build_judge_prompt,
    validate_judge_json,
    agreement,
    order_consistency,
)

display(pd.DataFrame(list(RUBRIC.items()), columns=["판정", "기준"]))
display(pd.DataFrame(FIXTURES)[["id", "evidence", "answer", "gold", "judge"]])

**확인과 해석**

supported는 제공 근거의 뒷받침이며 모든 사실을 검증했다는 뜻은 아닙니다.

## 2. 숫자 규칙의 한계

같은 숫자가 있어도 부정 표현이면 반대 의미입니다.

In [ ]:
for answer in ["수령 후 7일 이내 반품 가능", "수령 후 7일 이내에는 반품 불가"]:
    print(answer, "문자열 검사:", "7일" in answer)

**확인과 해석**

문자열 규칙은 경보로 쓰되 충실성 평가를 대체하지 않습니다.

## 3. 구조화된 평가 요청

질문·근거·답변을 분리하고 판정과 이유를 JSON으로 요구합니다.

In [ ]:
prompt = build_judge_prompt(
    "언제까지 반품 가능한가요?", FIXTURES[0]["evidence"], FIXTURES[0]["answer"]
)
print(prompt)

**확인과 해석**

프롬프트만 생성합니다. 구분자만으로 prompt injection 안전성이 보장되지 않습니다.

## 4. 결과 파싱

형식 오류를 임의 점수로 바꾸지 않습니다.

In [ ]:
print(validate_judge_json('{"verdict":"supported","reason":"7일 조건 일치"}'))
try:
    validate_judge_json('{"verdict":"좋음","reason":"그냥 좋아요"}')
except ValueError as error:
    print("형식 거부:", error)

**확인과 해석**

유효 JSON과 정확한 판정은 별도입니다.

## 5. 불일치 분석

평가자 결과도 평가합니다.

In [ ]:
a = agreement()
print("작성 예시 일치율:", a["agreement"])
display(pd.DataFrame(a["confusion"]).T)
display(pd.DataFrame(a["disagreement"])[["id", "gold", "judge", "reason"]])

**확인과 해석**

5/8=62.5%는 fixture 계산입니다. 실제 LLM 성능으로 발표하지 않습니다.

## 6. 위치 편향

위치를 바꿔도 같은 답변을 고르는지 확인합니다.

In [ ]:
pairs = [
    dict(winner_ab="answer1", winner_ba="answer1"),
    dict(winner_ab="answer1", winner_ba="answer2"),
    dict(winner_ab="tie", winner_ba="tie"),
]
print("작성 예시의 순서 일관성:", order_consistency(pairs))

**확인과 해석**

슬롯 A/B가 아닌 의미상의 answer ID를 비교합니다. 일관성과 정확성은 별개입니다.

## 확장 실험과 프로젝트 적용

1. 조건 누락·근거 부족·수치 오류를 추가합니다.
2. 선택 확장: 승인된 모델에 prompt를 전달하고 원응답·버전·설정·비용을 기록합니다. 기본 실습은 네트워크 호출이 없습니다.
3. 독립 검토와 불일치 조정을 설계합니다.

**실행 실패 시:** ModuleNotFoundError는 커널과 설치 터미널이 같은 .venv인지 확인합니다.
ROOT 오류는 저장소 안에서 파일을 열면 해결됩니다. 변수가 섞였다면 Restart Kernel 후 Run All.
예외 실험은 try/except로 설명하므로 정상 실행에서는 빨간 오류로 끝나지 않습니다.

**기록:** 변경 설정 / 예상 / 실제 관찰 / 실패 질문 ID / 다음 행동을 5줄로 적습니다.
개인 프로젝트 원문·연락처·API 키는 공개 GitHub에 올리지 않습니다.
